# STEP 3 — Axis 2: Calibration (ECE)

**What this step does:**
- Runs all 7 models on GSM8K + TruthfulQA with confidence elicitation appended to each prompt
- Both `direct` and `cot` modes
- Computes ECE and reliability diagram data per (model, mode, dataset)

**Outputs produced:**
- `results/calibration_raw.csv` — per-question answer + confidence + correct flag
- `results/calibration_ece.csv` — ECE per (model, mode, dataset)
- `results/calibration_bin_data.json` — reliability diagram data (bins)

**Estimated run time:** ~3–4 hours

**Prerequisite:** Step 1 must have completed

In [ ]:
import sys, json, gc
import torch
import pandas as pd
from pathlib import Path

sys.path.insert(0, '/kaggle/working/light_paper')

BASE_DIR    = Path('/kaggle/working/light_paper')
DATA_DIR    = BASE_DIR / 'data'
RESULTS_DIR = BASE_DIR / 'results'

from src.models    import MODEL_REGISTRY, load_model
from src.prompts   import make_calibration_prompt
from src.inference import generate_responses, parse_gsm8k_answer, parse_truthfulqa_answer, parse_confidence
from src.metrics   import compute_ece, compute_calibration_by_model

with open(DATA_DIR / 'datasets.json') as f:
    all_data = json.load(f)

# Calibration only on GSM8K + TruthfulQA (BBH answers are too short for verbalized confidence)
CAL_DATASETS = ['gsm8k', 'truthfulqa']
MODES        = ['direct', 'cot']

PARSERS = {'gsm8k': parse_gsm8k_answer, 'truthfulqa': parse_truthfulqa_answer}

print('Ready.')

In [ ]:
# ── Resume-safe ────────────────────────────────────────────────────────────
RAW_PATH = RESULTS_DIR / 'calibration_raw.csv'

if RAW_PATH.exists():
    existing_df = pd.read_csv(RAW_PATH)
    done_keys = set(zip(existing_df['model'], existing_df['mode'], existing_df['dataset']))
    print(f'⚡ Resuming — {len(done_keys)} combos done')
else:
    existing_df = pd.DataFrame()
    done_keys = set()

all_rows = existing_df.to_dict('records') if not existing_df.empty else []

In [ ]:
# ── Main calibration loop ─────────────────────────────────────────────────
for model_config in MODEL_REGISTRY:
    for mode in MODES:
        pending = [
            ds for ds in CAL_DATASETS
            if (model_config.name, mode, ds) not in done_keys
        ]
        if not pending:
            print(f'✓ Skip {model_config.name}/{mode}')
            continue

        print(f'\n── {model_config.name} | {mode} ──────────────────')
        model, tokenizer = load_model(model_config)

        for ds_name in pending:
            samples = all_data[ds_name]
            parser  = PARSERS[ds_name]

            prompts = [make_calibration_prompt(s['question'], mode, ds_name) for s in samples]
            print(f'  [{ds_name}] {len(prompts)} prompts...')

            responses = generate_responses(
                model, tokenizer, prompts,
                max_new_tokens=640,   # longer — needs answer + confidence line
                batch_size=4
            )

            for s, resp in zip(samples, responses):
                predicted  = parser(resp) or ''
                confidence = parse_confidence(resp)  # float 0–1 or None
                correct    = (
                    str(predicted).lower().strip() == str(s['answer']).lower().strip()
                )
                all_rows.append({
                    'model':        model_config.name,
                    'size_b':       model_config.size_b,
                    'mode':         mode,
                    'dataset':      ds_name,
                    'id':           s['id'],
                    'answer':       s['answer'],
                    'predicted':    predicted,
                    'confidence':   confidence,
                    'correct':      correct,
                    'raw_response': resp[:600],
                })

            # Save incrementally
            pd.DataFrame(all_rows).to_csv(RAW_PATH, index=False)
            done_keys.add((model_config.name, mode, ds_name))

            # Quick ECE preview
            valid = [r for r in all_rows
                     if r['model'] == model_config.name
                     and r['mode'] == mode
                     and r['dataset'] == ds_name
                     and r['confidence'] is not None]
            if valid:
                ece_res = compute_ece(
                    [r['confidence'] for r in valid],
                    [r['correct'] for r in valid]
                )
                conf_avg = sum(r['confidence'] for r in valid) / len(valid)
                print(f'  [{ds_name}] ECE={ece_res["ece"]:.3f}, mean_conf={conf_avg:.3f}, n_valid={len(valid)}')

        del model, tokenizer
        gc.collect()
        torch.cuda.empty_cache()
        print(f'  GPU freed')

In [ ]:
# ── Compute ECE summary + bin data ────────────────────────────────────────
df = pd.read_csv(RAW_PATH)
df['correct'] = df['correct'].astype(bool)

ece_summary = compute_calibration_by_model(df)
ece_summary.to_csv(RESULTS_DIR / 'calibration_ece.csv', index=False)

# Bin data for reliability diagrams
bin_data = {}
for (model, mode, dataset), group in df.groupby(['model', 'mode', 'dataset']):
    valid = group.dropna(subset=['confidence'])
    if len(valid) >= 10:
        result = compute_ece(valid['confidence'].tolist(), valid['correct'].tolist())
        bin_data[f'{model}/{mode}/{dataset}'] = result['bin_data']

with open(RESULTS_DIR / 'calibration_bin_data.json', 'w') as f:
    json.dump(bin_data, f, indent=2)

print('ECE Summary (CoT vs Direct):')
pivot = ece_summary.pivot_table(index=['model'], columns=['mode','dataset'], values='ece').round(3)
print(pivot.to_string())
print(f'\n✅ STEP 3 COMPLETE')
print(f'   Outputs: results/calibration_raw.csv, calibration_ece.csv, calibration_bin_data.json')
print(f'   Next: Run step4_consistency.ipynb')